In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

driver = webdriver.Chrome()
driver.maximize_window()
wait = WebDriverWait(driver, 10)

In [ ]:
try:
    driver.get("http://localhost:5173/")
    driver.execute_script("window.localStorage.clear(); window.sessionStorage.clear();")
    driver.get("http://localhost:5173/login")

    wait.until(EC.presence_of_element_located((By.ID, "username")))
    driver.find_element(By.ID, "username").send_keys("rafi")
    driver.find_element(By.ID, "password").send_keys("787878")
    driver.find_element(By.ID, "sign-in-btn").click()
    time.sleep(3)

    # Sidebar items verified in AppShell.jsx (plus pinned AI Assistant).
    # Per-item markers verified in their page components (see comments).
    targets = [
        ("Dashboard", "//section[@aria-label='Admin dashboard']"),  # DashboardContainer.jsx
        ("POS / Sales", "//h2[text()='POS / Sales']"),  # Sales.jsx
        ("Medicines & Inventory", "//h2[text()='Inventory']"),  # MedicinesInventoryPage.jsx
        ("Customers", "//h2[text()='Customers']"),  # CustomerDirectory.jsx
        ("CRM", "//nav[@aria-label='CRM sections']"),  # CRMModule.jsx
        ("Orders", "//h2[text()='Sales Orders']"),  # OrdersPage.jsx
        ("Reports", "//button[text()='30 Days']"),  # ReportsPage.jsx
        ("Notifications", "//*[text()='Notifications']"),  # NotificationsPage.jsx
        ("AI Assistant", "//*[@id='ai-chat-input']"),  # ChatInput.jsx
        ("Settings", "//*[text()='Full Name']"),  # SettingsPage.jsx
    ]
    for label, marker in targets:
        wait.until(EC.element_to_be_clickable((By.XPATH, f"//aside//button[contains(., '{label}')]"))).click()
        time.sleep(2)
        wait.until(EC.visibility_of_element_located((By.XPATH, marker)))
        assert driver.find_elements(By.XPATH, "//nav[@aria-label='Staff']"), "Staff sidebar lost."
        assert not driver.find_elements(By.ID, "username"), "Logged out to the login form."
        print(f"PASS: {label} navigation")

    # Admin-only link (verified: {isAdmin && ...} in AppShell.jsx) - presence only, do not leave the SPA
    um = driver.find_elements(By.XPATH, "//aside//a[@href='/admin/']")
    assert um and "User Management" in um[0].text
    print("PASS: User Management navigation (link present for Admin)")
except Exception as e:
    print("FAIL:", e)
    driver.save_screenshot("42_sidebar_navigation_FAIL.png")
finally:
    driver.quit()